# Notebook 03 - LangChain: Prompts, Chains, Document Loaders e Agents

## Tech Challenge Fase 3 - Assistente Virtual Medico

Este notebook cobre os fundamentos do LangChain para o projeto:
1. Prompt Engineering com LangChain
2. Chains (Cadeias) para processamento sequencial
3. Document Loaders para carregar prontuarios e protocolos
4. Agents para interacao com ferramentas externas

---
## 1. Configuracao do Ambiente

In [1]:
import os
import torch
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate

load_dotenv()

# Configuracao do Modelo
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_PATH = "../models/assistente_medico_final"

# FLAG: True = usar modelo fine-tuned, False = usar modelo base puro
USE_FINETUNED = True

# Carregar LLM usando modulo centralizado
import sys
sys.path.append('..')
from src.model_loader import load_llm

llm = load_llm(
    model_name=MODEL_NAME,
    adapter_path=ADAPTER_PATH,
    use_finetuned=USE_FINETUNED
)

print(f"Modelo carregado com sucesso!")
print(f"Modelo base: {MODEL_NAME}")
print(f"Modo: {'Fine-tuned' if USE_FINETUNED else 'Base'}")

/Users/rodrigofranco/Environment/FIAP/fase 3/aulas/projeto-assistente-medico/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Carregando modelo base...


Loading weights: 100%|██████████| 201/201 [00:04<00:00, 47.58it/s]


Aplicando adapter LoRA...


[transformers] Passing `generation_config` together with generation-related arguments=({'top_k', 'do_sample', 'pad_token_id', 'max_new_tokens', 'top_p', 'temperature', 'repetition_penalty', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Modelo carregado com sucesso!
Modelo base: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Modo: Fine-tuned


---
## 2. Prompt Engineering com LangChain

### Estrutura Basica de um Prompt

Um prompt eficaz deve conter:
- **Instrucao**: a tarefa que o modelo deve realizar
- **Contexto**: informacoes adicionais para entender a tarefa
- **Dados de entrada**: o conteudo ou questao a resolver
- **Indicador de saida**: formato ou tipo de resposta esperado

In [2]:
from langchain_core.prompts import PromptTemplate

# Prompt para assistente medico (formato TinyLlama)
template_medico = """
### Instruction:
Voce e um assistente medico especializado em {especialidade}.

REGRAS INEGOCIAVEIS:
1. Nao prescreva medicamentos diretamente
2. Nao faca diagnosticos definitivos
3. SEMPRE inclua: 'Esta resposta e uma sugestao e deve ser validada por um medico'
4. SEMPRE cite a fonte do protocolo utilizado
5. Se nao tiver certeza, diga 'Nao tenho informacao suficiente'

Analise a consulta do paciente e forneça:
1. Diagnostico diferencial
2. Exames complementares sugeridos
3. Conduta recomendada

Paciente: {nome}
Idade: {idade}
Sintomas: {sintomas}
Historico: {historico}

Responda de forma estruturada.

### Response:
"""

prompt = PromptTemplate(
    template=template_medico,
    input_variables=["especialidade", "nome", "idade", "sintomas", "historico"]
)

# Formatando o prompt
prompt_formatado = prompt.format(
    especialidade="Clinica Medica",
    nome="[Paciente]",
    idade="65 anos",
    sintomas="Dor toracica, dispneia, febre",
    historico="Hipertensao, Diabetes Mellitus tipo 2"
)

print("Prompt gerado:")
print(prompt_formatado)

Prompt gerado:

### Instruction:
Voce e um assistente medico especializado em Clinica Medica.

REGRAS INEGOCIAVEIS:
1. Nao prescreva medicamentos diretamente
2. Nao faca diagnosticos definitivos
3. SEMPRE inclua: 'Esta resposta e uma sugestao e deve ser validada por um medico'
4. SEMPRE cite a fonte do protocolo utilizado
5. Se nao tiver certeza, diga 'Nao tenho informacao suficiente'

Analise a consulta do paciente e forneça:
1. Diagnostico diferencial
2. Exames complementares sugeridos
3. Conduta recomendada

Paciente: [Paciente]
Idade: 65 anos
Sintomas: Dor toracica, dispneia, febre
Historico: Hipertensao, Diabetes Mellitus tipo 2

Responda de forma estruturada.

### Response:



In [3]:
# Prompt para classificacao de urgencia (formato TinyLlama)
template_urgencia = """
### Instruction:
Classifique a urgencia da seguinte consulta medica em uma das categorias:
- VERMELHA (Emergencia - acao imediata)
- LARANJA (Muito urgente - atendimento em ate 30 min)
- AMARELA (Urgente - atendimento em ate 2h)
- VERDE (Pouco urgente - atendimento em ate 4h)
- AZUL (Nao urgente - agendamento normal)

Consulta: {consulta}

Forneça:
1. Classificacao de cor
2. Justificativa
3. Conduta recomendada

### Response:
"""

prompt_urgencia = PromptTemplate(
    template=template_urgencia,
    input_variables=["consulta"]
)

print("Prompt de urgencia criado!")

Prompt de urgencia criado!


---
## 3. Chains (Cadeias)

### 3.1 LLMChain - Cadeia Basica

A LLMChain conecta um prompt a um LLM, formando a unidade basica do LangChain.

In [4]:
# Cadeia simples para analise clinica (formato moderno LCEL)
chain_analise = prompt | llm

# Executar a cadeia
resultado = chain_analise.invoke({
    "especialidade": "Clinica Medica",
    "nome": "[Paciente]",
    "idade": "65 anos",
    "sintomas": "Dor toracica, dispneia, febre",
    "historico": "Hipertensao, Diabetes Mellitus tipo 2"
})

print("Resultado da analise:")
print(resultado)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Resultado da analise:

### Instruction:
Voce e um assistente medico especializado em Clinica Medica.

REGRAS INEGOCIAVEIS:
1. Nao prescreva medicamentos diretamente
2. Nao faca diagnosticos definitivos
3. SEMPRE inclua: 'Esta resposta e uma sugestao e deve ser validada por um medico'
4. SEMPRE cite a fonte do protocolo utilizado
5. Se nao tiver certeza, diga 'Nao tenho informacao suficiente'

Analise a consulta do paciente e forneça:
1. Diagnostico diferencial
2. Exames complementares sugeridos
3. Conduta recomendada

Paciente: [Paciente]
Idade: 65 anos
Sintomas: Dor toracica, dispneia, febre
Historico: Hipertensao, Diabetes Mellitus tipo 2

Responda de forma estruturada.

### Response:
Diagnosticodiferencial:
- Hipertensao arterial
- Diabetes Mellitus tipo 2

Exames complementares sugeridos:
- Fisiopatologia cardiovascular
- ECG
- Echocardiograma
- Atenuacao de pressao

Conduta recomendada:
- Atenuacao de pressao
- Dieta saudavel
- Exercicio regular
- Atenuacao de alcoolismo
- Terapia

### 3.2 SequentialChain - Pipeline Sequencial

Permite encadear multiplas etapas de processamento.

In [5]:
from langchain_core.runnables import RunnableLambda

# Funcao de classificacao
def classificar_consulta(consulta):
    prompt_classificacao = PromptTemplate(
        template="""### Instruction:
Classifique a consulta medica em uma das categorias: CLINICA GERAL, CARDIOLOGIA, PNEUMOLOGIA, NEUROLOGIA, URGENCIA. Consulta: {consulta}. Responda APENAS com o nome da categoria.

### Response:
""",
        input_variables=["consulta"]
    )
    chain = prompt_classificacao | llm
    resultado = chain.invoke({"consulta": consulta})
    return resultado

# Funcao de protocolo
def buscar_protocolo(dados):
    classificacao = dados["classificacao"]
    consulta = dados["consulta"]
    prompt_protocolo = PromptTemplate(
        template="""### Instruction:
Com base na classificacao {classificacao} e na consulta original: {consulta}. Sugira o protocolo medico adequado com: 1. Condutas principais, 2. Exames complementares, 3. Medidas de urgencia se aplicavel.

### Response:
""",
        input_variables=["classificacao", "consulta"]
    )
    chain = prompt_protocolo | llm
    resultado = chain.invoke({"classificacao": classificacao, "consulta": consulta})
    return resultado

# Funcao de resposta final
def gerar_resposta(dados):
    prompt_resposta = PromptTemplate(
        template="""### Instruction:
Gere uma resposta completa e estruturada ao medico. Classificacao: {classificacao}. Protocolo sugerido: {protocolo}. Inclua: 1. Resumo da analise, 2. Condutas recomendadas, 3. Exames solicitados, 4. Aviso de seguranca (validacao humana obrigatoria).

### Response:
""",
        input_variables=["classificacao", "protocolo"]
    )
    chain = prompt_resposta | llm
    resultado = chain.invoke({"classificacao": dados["classificacao"], "protocolo": dados["protocolo"]})
    return resultado

# Funcao principal que encadeia tudo
def cadeia_sequencial(consulta):
    classificacao = classificar_consulta(consulta)
    print(f"Classificacao: {classificacao}")
    
    protocolo = buscar_protocolo({"classificacao": classificacao, "consulta": consulta})
    print(f"Protocolo gerado.")
    
    resposta = gerar_resposta({"classificacao": classificacao, "protocolo": protocolo})
    return resposta

print("Cadeia sequencial configurada!")
print("Fluxo: Classificacao -> Protocolo -> Resposta")

Cadeia sequencial configurada!
Fluxo: Classificacao -> Protocolo -> Resposta


In [6]:
# Executar a cadeia sequencial
resultado_sequencial = cadeia_sequencial(
    "Paciente de 65 anos com dor toracica intensa, sudorese e nausea. Historico de hipertensao e diabetes."
)

print("\n" + "="*60)
print("RESPOSTA FINAL DO ASSISTENTE:")
print("="*60)
print(resultado_sequencial)

Classificacao: ### Instruction:
Classifique a consulta medica em uma das categorias: CLINICA GERAL, CARDIOLOGIA, PNEUMOLOGIA, NEUROLOGIA, URGENCIA. Consulta: Paciente de 65 anos com dor toracica intensa, sudorese e nausea. Historico de hipertensao e diabetes.. Responda APENAS com o nome da categoria.

### Response:
CATEGORIA: PNEUMOLOGIA
Protocolo gerado.

RESPOSTA FINAL DO ASSISTENTE:
### Instruction:
Gere uma resposta completa e estruturada ao medico. Classificacao: ### Instruction:
Classifique a consulta medica em uma das categorias: CLINICA GERAL, CARDIOLOGIA, PNEUMOLOGIA, NEUROLOGIA, URGENCIA. Consulta: Paciente de 65 anos com dor toracica intensa, sudorese e nausea. Historico de hipertensao e diabetes.. Responda APENAS com o nome da categoria.

### Response:
CATEGORIA: PNEUMOLOGIA. Protocolo sugerido: ### Instruction:
Com base na classificacao ### Instruction:
Classifique a consulta medica em uma das categorias: CLINICA GERAL, CARDIOLOGIA, PNEUMOLOGIA, NEUROLOGIA, URGENCIA. Consu

---
## 4. Document Loaders

### 4.1 Carregando Documentos Medicos

Document Loaders permitem carregar diferentes formatos de documentos para uso com LLMs.

In [7]:
# Criar documentos de exemplo para demonstracao
import json
import os

# Criar diretorio de protocolos
protocolos_dir = "../data/protocolos"
os.makedirs(protocolos_dir, exist_ok=True)

# Criar arquivo de protocolo de exemplo
protocolo_exemplo = """
# PROTOCOLO DE PNEUMONIA HOSPITALAR

## OBJETIVO
Estabelecer diretrizes para diagnostico e tratamento de pneumonia adquirida em ambiente hospitalar.

## DEFINICAO
Pneumonia adquirida apos 48 horas de internacao, nao associada a ventilação mecanica.

## DIAGNOSTICO
### Sinais e Sintomas
- Febre > 38°C ou hipotermia < 36°C
- Tosse com expectoração purulenta
- Dispneia progressiva
- Dor toracica pleuritica
- Alteracao do estado mental em idosos

### Exames Complementares
- Raio-X de torax (infiltrado pulmonar novo ou progressivo)
- Hemograma completo
- PCR e Procalcitonina
- Gasometria arterial
- Culturas (hemocultiva, ESC, BAL)

## TRATAMENTO
### Antibioticoterapia Empirica
- Ceftriaxona 2g IV 24h + Azitromicina 500mg IV 24h
ou
- Piperacilina-Tazobactam 4.5g IV 8h + Azitromicina 500mg IV 24h

### Reavaliacao
- Clinica em 48-72 horas
- Ajuste conforme antibiograma (despistagem)
- Duracao minima: 7 dias

## PREVENCAO
- Higiene das maos
- Elevacao da cabeceira 30-45 graus
- Curatismo oral diario
- Desmame precoce da ventilação mecanica
"""

with open(os.path.join(protocolos_dir, "protocolo_pneumonia.txt"), 'w') as f:
    f.write(protocolo_exemplo)

print("Protocolo de exemplo criado!")

Protocolo de exemplo criado!


In [8]:
# Carregar documento de texto
from langchain_community.document_loaders import TextLoader

loader = TextLoader(os.path.join(protocolos_dir, "protocolo_pneumonia.txt"), encoding='utf-8')
documentos = loader.load()

print(f"Documentos carregados: {len(documentos)}")
print(f"Tamanho: {len(documentos[0].page_content)} caracteres")
print(f"\nPrimeiros 300 caracteres:")
print(documentos[0].page_content[:300])

/var/folders/kq/5x85h13n5h3217kfty7zt1h80000gn/T/ipykernel_67642/983703252.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Documentos carregados: 1
Tamanho: 1048 caracteres

Primeiros 300 caracteres:

# PROTOCOLO DE PNEUMONIA HOSPITALAR

## OBJETIVO
Estabelecer diretrizes para diagnostico e tratamento de pneumonia adquirida em ambiente hospitalar.

## DEFINICAO
Pneumonia adquirida apos 48 horas de internacao, nao associada a ventilação mecanica.

## DIAGNOSTICO
### Sinais e Sintomas
- Febre > 38


In [9]:
# Carregar multiplos documentos
from langchain_community.document_loaders import DirectoryLoader

loader_diretorio = DirectoryLoader(
    protocolos_dir,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

docs = loader_diretorio.load()
print(f"Total de documentos carregados: {len(docs)}")

Total de documentos carregados: 1


---
## 5. Text Splitters

Documentos longos precisam ser divididos em partes menores para processamento eficiente.

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Dividir documento em chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documentos)

print(f"Total de chunks: {len(chunks)}")
print(f"\nExemplo de chunk:")
print(chunks[0].page_content[:200])

Total de chunks: 3

Exemplo de chunk:
# PROTOCOLO DE PNEUMONIA HOSPITALAR

## OBJETIVO
Estabelecer diretrizes para diagnostico e tratamento de pneumonia adquirida em ambiente hospitalar.

## DEFINICAO
Pneumonia adquirida apos 48 horas de 


---
## 6. Agents

### 6.1 Conceito de Agents

Agents utilizam LLMs para decidir quais ferramentas usar e em que ordem. O padrao ReAct (Reasoning + Acting) e o mais comum:

1. **Thought** (Pensamento): O raciocinio do agente
2. **Action** (Acao): A ferramenta a ser utilizada
3. **Observation** (Observacao): O resultado da acao

In [11]:
# Definir ferramentas (tools)
def buscar_protocolo_agente(condicao: str) -> str:
    """Busca protocolo medico para uma condicao."""
    protocolos = {
        "pneumonia": "Protocolo de Pneumonia Hospitalar: Ceftriaxona 2g + Azitromicina 500mg IV",
        "sepse": "Protocolo de Sepse: Resuscitacao 30mL/kg + Antibiotico em 1h",
        "avc": "Protocolo de AVC: Trombolise ate 4.5h + Trombectomia ate 24h",
        "insuficiencia cardiaca": "Protocolo de IC: Furosemida 40-80mg IV + Oxigenoterapia"
    }
    condicao_lower = condicao.lower()
    for chave, valor in protocolos.items():
        if chave in condicao_lower:
            return valor
    return "Protocolo nao encontrado. Consulte o servico de referencia."

def calcular_risco_ashburner(pontuacao: int) -> str:
    """Calcula risco baseado na pontuacao de Ashburner (simplificado)."""
    pontuacao = int(pontuacao)
    if pontuacao >= 6:
        return "Risco ALTO. Internacao recomendada."
    elif pontuacao >= 3:
        return "Risco MODERADO. Observacao por 24h."
    else:
        return "Risco BAIXO. Alta ambulatorial possivel."

# Funcao do agente que usa as ferramentas
def agente_medico(pergunta: str) -> str:
    """Agente simples que direciona para a ferramenta correta."""
    pergunta_lower = pergunta.lower()
    
    # Verificar se e busca de protocolo
    if "protocolo" in pergunta_lower:
        # Extrair condicao da pergunta
        for condicao in ["pneumonia", "sepse", "avc", "insuficiencia cardiaca"]:
            if condicao in pergunta_lower:
                return buscar_protocolo_agente(condicao)
        return "Para qual condicao voce deseja o protocolo?"
    
    # Verificar se e calculo de risco
    if "risco" in pergunta_lower or "pontuacao" in pergunta_lower:
        # Extrair numero da pergunta
        import re
        numeros = re.findall(r'\d+', pergunta)
        if numeros:
            return calcular_risco_ashburner(numeros[0])
        return "Qual a pontuacao do paciente?"
    
    # Se nao for nenhuma das anteriores, usar o LLM
    prompt = PromptTemplate(
        template="""### Instruction:
Voce e um assistente medico. Responda: {pergunta}

### Response:
""",
        input_variables=["pergunta"]
    )
    chain = prompt | llm
    resultado = chain.invoke({"pergunta": pergunta})
    return resultado

print("Agente medico configurado!")
print("Ferramentas disponiveis:")
print("  - Buscar_Protocolo: Busca protocolo medico para uma condicao clinica")
print("  - Calcular_Risco: Calcula nivel de risco do paciente baseado em pontuacao")

Agente medico configurado!
Ferramentas disponiveis:
  - Buscar_Protocolo: Busca protocolo medico para uma condicao clinica
  - Calcular_Risco: Calcula nivel de risco do paciente baseado em pontuacao


In [12]:
# Testar o agente
print("\n" + "="*60)
print("TESTE DO AGENTE MEDICO")
print("="*60)

# Teste 1: Busca de protocolo
resultado1 = agente_medico("Qual protocolo para pneumonia?")
print(f"\nResposta 1: {resultado1}")

# Teste 2: Calculo de risco
resultado2 = agente_medico("Paciente com pontuacao 7. Qual o risco?")
print(f"\nResposta 2: {resultado2}")


TESTE DO AGENTE MEDICO

Resposta 1: Protocolo de Pneumonia Hospitalar: Ceftriaxona 2g + Azitromicina 500mg IV

Resposta 2: Risco ALTO. Internacao recomendada.


---
## 7. Output Parsers

Output Parsers estruturam as respostas do LLM em formatos utilizaveis.

In [13]:
import json

# Exemplo de Output Parser simples usando JSON
def parsear_resposta_json(texto: str) -> dict:
    """Extrai JSON de uma resposta do LLM."""
    try:
        # Tenta encontrar JSON no texto
        inicio = texto.find('{')
        fim = texto.rfind('}') + 1
        if inicio != -1 and fim != 0:
            return json.loads(texto[inicio:fim])
        return {"erro": "JSON nao encontrado"}
    except json.JSONDecodeError:
        return {"erro": "Formato invalido"}

# Prompt com instrucoes de formatacao
prompt_json = PromptTemplate(
    template="""### Instruction:
Analise a consulta e responda em formato JSON com as chaves:
- diagnostico: diagnostico diferencial
- exames: exames complementares
- conduta: conduta recomendada
- urgencia: baixo/medio/alto

Consulta: {consulta}

Responda APENAS com o JSON:

### Response:
""",
    input_variables=["consulta"]
)

# Testar parser
chain_json = prompt_json | llm
resultado = chain_json.invoke({"consulta": "Paciente com febre e tosse ha 3 dias"})
dados = parsear_resposta_json(resultado)

print("Resposta estruturada:")
print(json.dumps(dados, indent=2, ensure_ascii=False))

Resposta estruturada:
{
  "diagnostico": [
    {
      "id": 1,
      "descricao": "Febre"
    },
    {
      "id": 2,
      "descricao": "Tosse"
    }
  ],
  "exames": [
    {
      "id": 1,
      "descricao": "Exame de urina"
    },
    {
      "id": 2,
      "descricao": "Exame de fígado"
    },
    {
      "id": 3,
      "descricao": "Exame de neurologia"
    }
  ],
  "conduta": [
    {
      "id": 1,
      "descricao": "Atenção primária"
    },
    {
      "id": 2,
      "descricao": "Emergencia"
    },
    {
      "id": 3,
      "descricao": "Agente antibiotico"
    }
  ],
  "urgencia": [
    {
      "id": 1,
      "descricao": "Baixo"
    },
    {
      "id": 2,
      "descricao": "Medio"
    },
    {
      "id": 3,
      "descricao": "Alto"
    }
  ]
}


---
## 8. Resumo

### Componentes do LangChain Utilizados:

| Componente | Funcao | Exemplo no Projeto |
|------------|--------|--------------------|
| **PromptTemplate** | Estruturar prompts | Templates para consultas medicas |
| **LLMChain** | Conectar prompt + LLM | Analise clinica |
| **SequentialChain** | Encadear multiplas etapas | Classificacao -> Protocolo -> Resposta |
| **Document Loaders** | Carregar documentos | Protocolos medicos |
| **Text Splitters** | Dividir documentos longos | Chunking de protocolos |
| **Agents** | Usar ferramentas externas | Busca de protocolos |
| **Output Parsers** | Estruturar respostas | Formatacao medica |

### Proximo notebook:
O notebook `04_langgraph_rag_medico.ipynb` integrara o LangChain com LangGraph e RAG para criar o fluxo completo.

In [14]:
print("\n=== NOTEBOOK 03 CONCLUIDO ===")
print("Fundamentos do LangChain dominados!")


=== NOTEBOOK 03 CONCLUIDO ===
Fundamentos do LangChain dominados!
